# Water level prediction using ERA5 (Keras baseline)

This notebook:
- Downloads the dataset
- Normalizes ERA5 data
- Builds features (ephemerides + time + ERA5 + lat/lon)
- Trains a neural network (MLP)
- Evaluates RMSE on test set (2020)

**Constraint respected:** no past water level values are used as inputs.


In [110]:
!pip -q install gdown tensorflow

In [111]:
import os
import numpy as np
import tensorflow as tf
from tqdm import tqdm

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Download data

In [112]:
import gdown

files = {
    # train
    "wl_2010-2019.npy": "1Ncexf_vB55cpiCeNr-hIRrdpquYaav6B",
    "dist_alt_az_moon-sun_coord13-45_2010-2019_norm.npy": "1THbGvO9mVjg_wfZTabRbQBOVpaECl3my",
    "ERA5_adriatic_u10v10sp_2010-2019.npy": "16M1zB54PKkKS6SK8W_U83UURWu1T_AxR",
    "tvec_2010-2019.npy": "161OYs8KQSn3RrXezCNwFvOewXLJe5wBf",
    # test
    "wl_2020.npy": "1iwqd4xzHc98OYqBpGsuUW4SG5AQDtdNR",
    "dist_alt_az_moon-sun_coord13-45_2020_norm.npy": "1cHqyeXtmaiC_3v9uadMD7Y0hONGf2R1q",
    "ERA5_adriatic_u10v10sp_2020.npy": "1AoFAD2viMarikhU5b5Etdsklx08EzKKb",
    "tvec_2020.npy": "1sWoTlJih-mqDdP9TBzhyXTqHHS5Jr9fe",
    # coords
    "lat.npy": "1Mg52QAIo4bfpzJF0dsI8mpZf09tHTzj8",
    "lon.npy": "1wWz0EWbGiBkZ0vfJD8KeZkmVZfmRiYLk",
}

for fname, fid in files.items():
    if not os.path.exists(fname):
        print("Downloading", fname)
        gdown.download(id=fid, output=fname, quiet=False)
    else:
        print("Exists:", fname)

Exists: wl_2010-2019.npy
Exists: dist_alt_az_moon-sun_coord13-45_2010-2019_norm.npy
Exists: ERA5_adriatic_u10v10sp_2010-2019.npy
Exists: tvec_2010-2019.npy
Exists: wl_2020.npy
Exists: dist_alt_az_moon-sun_coord13-45_2020_norm.npy
Exists: ERA5_adriatic_u10v10sp_2020.npy
Exists: tvec_2020.npy
Exists: lat.npy
Exists: lon.npy


## Load data (memory-mapped)

In [113]:
lat = np.load("lat.npy")
lon = np.load("lon.npy")

train_wl = np.load("wl_2010-2019.npy", mmap_mode="r")
train_ephem = np.load("dist_alt_az_moon-sun_coord13-45_2010-2019_norm.npy", mmap_mode="r")
train_era5 = np.load("ERA5_adriatic_u10v10sp_2010-2019.npy", mmap_mode="r")
train_tvec = np.load("tvec_2010-2019.npy", mmap_mode="r")

test_wl = np.load("wl_2020.npy", mmap_mode="r")
test_ephem = np.load("dist_alt_az_moon-sun_coord13-45_2020_norm.npy", mmap_mode="r")
test_era5 = np.load("ERA5_adriatic_u10v10sp_2020.npy", mmap_mode="r")
test_tvec = np.load("tvec_2020.npy", mmap_mode="r")

T_train = train_wl.shape[0]
N_nodes = train_wl.shape[1]
print("Train shapes:", train_wl.shape, train_ephem.shape, train_era5.shape, train_tvec.shape)
print("Test shapes:", test_wl.shape, test_ephem.shape, test_era5.shape, test_tvec.shape)

Train shapes: (87648, 5000) (6, 87648) (3, 87648, 5, 9) (6, 87648)
Test shapes: (8784, 5000) (6, 8784) (3, 8784, 5, 9) (6, 8784)


## Utilities

In [114]:
def get_era5_coord(lat, lon):
    """
    Match node coordinates to closest ERA5 grid index (safe bounds).
    """
    era5_row, era5_col = 5, 9
    lat_min, lat_max = 44.94972, 45.8
    lon_min, lon_max = 12.12863, 13.81283

    delta_lat = lat_max - lat_min
    delta_lon = lon_max - lon_min

    lon_coord = np.ceil((lon - lon_min) / delta_lon * (era5_col - 1))
    lat_coord = 4 - np.ceil((lat - lat_min) / delta_lat * (era5_row - 1))

    lon_coord = np.clip(lon_coord, 0, era5_col - 1)
    lat_coord = np.clip(lat_coord, 0, era5_row - 1)

    return int(lat_coord), int(lon_coord)

def RMSE(y_true, y_pred):
    return np.sqrt(np.mean((y_pred - y_true) ** 2))

## Normalize ERA5 (using training statistics)

In [115]:
era5_mean = train_era5.mean(axis=(1,2,3), keepdims=True)
era5_std  = train_era5.std(axis=(1,2,3), keepdims=True) + 1e-6

def norm_era5(x):
    return (x - era5_mean) / era5_std

train_era5_norm = norm_era5(train_era5)
test_era5_norm  = norm_era5(test_era5)

print("Train ERA5 norm mean:", train_era5_norm.mean(axis=(1,2,3)))
print("Train ERA5 norm std:", train_era5_norm.std(axis=(1,2,3)))

Train ERA5 norm mean: [-1.43255746e-17 -1.91391983e-17  2.22948601e-15]
Train ERA5 norm std: [0.99999958 0.9999996  1.        ]


## Time encoding

In [116]:
def time_encoding(tvec):
    years  = tvec[0].astype(int)
    months = tvec[1].astype(int)
    days   = tvec[2].astype(int)
    hours  = tvec[3].astype(int)

    leap = (years % 4 == 0) & ((years % 100 != 0) | (years % 400 == 0))
    month_days = np.array([31,28,31,30,31,30,31,31,30,31,30,31])
    cum_days = np.concatenate([[0], np.cumsum(month_days)])

    doy = cum_days[months - 1] + days
    doy += leap & (months > 2)

    hour_sin = np.sin(2 * np.pi * hours / 24.0)
    hour_cos = np.cos(2 * np.pi * hours / 24.0)
    doy_sin = np.sin(2 * np.pi * doy / 365.0)
    doy_cos = np.cos(2 * np.pi * doy / 365.0)

    return np.stack([hour_sin, hour_cos, doy_sin, doy_cos], axis=1)

train_time_feat = time_encoding(train_tvec)
test_time_feat  = time_encoding(test_tvec)
print("Time features:", train_time_feat.shape)

Time features: (87648, 4)


## Precompute ERA5 grid mapping for nodes

In [117]:
node_era5 = np.array([get_era5_coord(lat[i], lon[i]) for i in range(N_nodes)])
print(node_era5.shape, node_era5[:5])

(5000, 2) [[4 8]
 [4 8]
 [4 8]
 [4 8]
 [4 8]]


## Feature builder

In [118]:
def build_features_pairs(t_idx, n_idx, ephem, time_feat, era5_norm):
    """Aligned pairs: t_idx[k], n_idx[k] -> one sample."""
    t_idx = np.asarray(t_idx)
    n_idx = np.asarray(n_idx)

    ep = ephem[:, t_idx].T
    tf = time_feat[t_idx]
    rows, cols = node_era5[n_idx].T
    er = era5_norm[:, t_idx, rows, cols].T
    latlon = np.stack([lat[n_idx], lon[n_idx]], axis=1)

    return np.concatenate([ep, tf, er, latlon], axis=1)

def build_features_grid(t_idx, n_idx, ephem, time_feat, era5_norm):
    """All combinations of t_idx and n_idx."""
    t_idx = np.asarray(t_idx)
    n_idx = np.asarray(n_idx)

    tt = np.repeat(t_idx, len(n_idx))
    nn = np.tile(n_idx, len(t_idx))

    feats = build_features_pairs(tt, nn, ephem, time_feat, era5_norm)
    return feats, tt, nn

## Train/validation split
- Use last 8760 hours (2019) for validation.

In [119]:
val_hours = 8760
train_idx = np.arange(0, T_train - val_hours)
val_idx = np.arange(T_train - val_hours, T_train)
print("Train hours:", len(train_idx), "Val hours:", len(val_idx))

Train hours: 78888 Val hours: 8760


## Model (Keras MLP)

In [120]:
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, Rescaling
from tensorflow.keras.models import Model

def get_model(input_dim):
    x = Input(shape=(input_dim,))
    x = Dense(128, activation="relu")(x)
    x = Dense(128, activation="relu")(x)
    y = Dense(1)(x)
    return tf.keras.Model(x, y)

# For your features: 15 values -> (15, 1)
model = get_model((15, 1))
model.compile(optimizer='adam', loss='mse')

ValueError: Invalid dtype: tuple

## Training loop (random sampling)

In [ ]:
def sample_batch(batch_size, time_pool):
    t_idx = np.random.choice(time_pool, size=batch_size, replace=True)
    n_idx = np.random.randint(0, N_nodes, size=batch_size)
    X = build_features_pairs(t_idx, n_idx, train_ephem, train_time_feat, train_era5_norm)
    y = train_wl[t_idx, n_idx]
    return X.astype(np.float32), y.astype(np.float32)

def eval_rmse_sample(num_samples=50000):
    t_idx = np.random.choice(val_idx, size=num_samples, replace=True)
    n_idx = np.random.randint(0, N_nodes, size=num_samples)
    X = build_features_pairs(t_idx, n_idx, train_ephem, train_time_feat, train_era5_norm)
    y = train_wl[t_idx, n_idx]
    pred = model.predict(X, verbose=0).squeeze()
    return RMSE(y, pred)

epochs = 8
steps_per_epoch = 400
batch_size = 2048

for epoch in range(1, epochs + 1):
    pbar = tqdm(range(steps_per_epoch), desc=f"Epoch {epoch}")
    for _ in pbar:
        X, y = sample_batch(batch_size, train_idx)
        loss = model.train_on_batch(X, y)
        pbar.set_postfix(loss=float(loss))

    val_rmse = eval_rmse_sample()
    print(f"Validation RMSE (sampled): {val_rmse:.4f}")

Epoch 1: 100%|██████████| 400/400 [00:06<00:00, 62.47it/s, loss=0.0447] 


Validation RMSE (sampled): 0.1860


Epoch 2: 100%|██████████| 400/400 [00:02<00:00, 151.67it/s, loss=0.0367]


Validation RMSE (sampled): 0.1801


Epoch 3: 100%|██████████| 400/400 [00:02<00:00, 143.57it/s, loss=0.0329]


...

Validation RMSE (sampled): 0.1565


Epoch 7: 100%|██████████| 400/400 [00:02<00:00, 144.08it/s, loss=0.0275]


Validation RMSE (sampled): 0.1539


Epoch 8: 100%|██████████| 400/400 [00:02<00:00, 144.90it/s, loss=0.0268]


Validation RMSE (sampled): 0.1556


## Full test RMSE (batched)

In [ ]:
def compute_test_rmse(time_block=24, node_block=500):
    sq_sum = 0.0
    count = 0

    for t0 in tqdm(range(0, test_wl.shape[0], time_block), desc="Test time blocks"):
        t_idx = np.arange(t0, min(t0 + time_block, test_wl.shape[0]))
        for n0 in range(0, N_nodes, node_block):
            n_idx = np.arange(n0, min(n0 + node_block, N_nodes))

            X, tt, nn = build_features_grid(t_idx, n_idx, test_ephem, test_time_feat, test_era5_norm)
            pred = model.predict(X, verbose=0).squeeze()

            y_true = test_wl[tt, nn]
            sq_sum += np.sum((pred - y_true) ** 2)
            count += len(pred)

    return np.sqrt(sq_sum / count)

test_rmse = compute_test_rmse(time_block=24, node_block=500)
print(f"Final Test RMSE: {test_rmse:.4f}")

Test time blocks:  57%|█████▋    | 209/366 [24:46<18:36,  7.11s/it]


KeyboardInterrupt: 

## Persistence baseline (reference)

In [ ]:
baseline = RMSE(test_wl[:-1], test_wl[1:])
print(f"Persistence baseline RMSE: {baseline:.4f}")